## Compare whether adding teams adds any value to the model

In [1]:
# import the libraries
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# select numeric columns and columns with team names
# ROOT = Path(__file__).resolve().parents[1]
df = pd.read_csv("../data/processed/reduced/Final_data.csv")

NUM = ["target_runs", "runs_left", "CRR", "balls_left",
       "RRR", "wickets_left"]
TEAMS = ["team_batting", "team_bowling"]


Now we build functions to test whether our model with teams included in it gives more better results


In [3]:
def build(use_teams):
    transformers = [("num", StandardScaler(), NUM)]
    if use_teams:
        transformers.append(
            # encode the string names to so the model can utilize them
            ("team", OneHotEncoder(handle_unknown="ignore"), TEAMS))
    return Pipeline([
        ("prep", ColumnTransformer(transformers)),
        ("model", LogisticRegression(max_iter=2000)),
    ])

def evaluate(use_teams):
    X, y, groups = df[NUM + TEAMS], df["won"], df["match_id"]
    scores = []
    for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
        pipe = build(use_teams).fit(X.iloc[tr], y.iloc[tr])
        p = pipe.predict_proba(X.iloc[te])[:, 1]
        scores.append((log_loss(y.iloc[te], p),
                       brier_score_loss(y.iloc[te], p),
                       accuracy_score(y.iloc[te], p > 0.5)))
    return np.mean(scores, axis=0)

def fold_logloss(use_teams, C):
    X, y, groups = df[NUM + TEAMS], df["won"], df["match_id"]
    out = []
    for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
        pipe = build(use_teams)
        pipe.set_params(model__C=C)
        pipe.fit(X.iloc[tr], y.iloc[tr])
        out.append(log_loss(y.iloc[te], pipe.predict_proba(X.iloc[te])[:, 1]))
    return np.array(out)

base = fold_logloss(False, 1.0)
print("no teams    ", base.round(4), base.mean().round(4))
for C in [1.0, 0.1, 0.01]:
    t = fold_logloss(True, C)
    print(f"teams C={C:<5}", t.round(4), t.mean().round(4),
          "worse in", (t > base).sum(), "of 5 folds")

no teams     [0.4609 0.4039 0.4077 0.4445 0.4697] 0.4373
teams C=1.0   [0.4693 0.4087 0.4083 0.4643 0.487 ] 0.4475 worse in 5 of 5 folds
teams C=0.1   [0.469  0.4085 0.4087 0.4638 0.4865] 0.4473 worse in 5 of 5 folds
teams C=0.01  [0.4696 0.4092 0.4121 0.4615 0.4834] 0.4471 worse in 5 of 5 folds
